In [1]:
# Title: UNDP People’s Climate Vote 2024 – Predictive Modelling and Country Clustering

# Objective
# This project analyses the 2024 UNDP People’s Climate Vote dataset to:

# • Predict which demographic groups show high support for strengthening climate commitments
# • Compare multiple classification models
# • Evaluate model performance using balanced accuracy, F1 score and ROC AUC
# • Identify country typologies using K-Means clustering
# • Visualise global patterns using PCA

# Dataset
# Official UNDP People’s Climate Vote 2024 survey
# 45,784 aggregated responses
# 73 countries
# 15 survey questions

# Methodology
# • Remove aggregate categories such as “All Ages” and “Global”
# • Transform data from long to wide format
# • Create binary target via median split of support for strengthening
# • Compare Logistic Regression, Random Forest, Gradient Boosting
# • Perform cross-validation
# • Conduct country-level clustering

# Key Outputs
# • Model comparison metrics
# • Feature importance analysis
# • Country cluster assignments
# • PCA cluster visualisation

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, 
                             f1_score, 
                             balanced_accuracy_score,
                             classification_report, 
                             confusion_matrix, 
                             roc_auc_score
)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings("ignore")

import os
os.environ["OMP_NUM_THREADS"] = "1"

In [3]:
# Visualization settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load dataset
DATA_PATH = "Peoples_Climate_Vote_Database_2024.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

# Validate required columns
required_columns = [
    "Country",
    "Age",
    "Education",
    "Question Text (Short)",
    "Response",
    "Weighted Mean",
    "QID"
]
missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

Dataset shape: (45784, 17)


In [4]:
# Remove aggregate categories
initial_rows = len(df)

df_filtered = df[df["Age"] != "All Ages"].copy()
df_filtered = df_filtered[df_filtered["Country"] != "Global"].copy()

filtered_rows = len(df_filtered)

# Determine education breakdown
edu_counts = (
    df_filtered
    .groupby(["Country", "Age"])["Education"]
    .nunique()
    .reset_index(name="n_edu_levels")
)

use_education = (edu_counts["n_edu_levels"] > 1).mean() > 0.3

if use_education:
    df_filtered = df_filtered[df_filtered["Education"] != "All Education"].copy()
    group_cols = ["Country", "Age", "Education"]
else:
    group_cols = ["Country", "Age"]

n_groups = df_filtered.groupby(group_cols).ngroups

print(f"Rows before filtering: {initial_rows}")
print(f"Rows after filtering: {filtered_rows}")
print(f"Number of respondent groups: {n_groups}")
print(f"Using education breakdown: {use_education}")

Rows before filtering: 45784
Rows after filtering: 20377
Number of respondent groups: 287
Using education breakdown: False


In [5]:
# Identify target question: strengthen / weaken commitments

mask = df_filtered["Question Text"].str.contains(
    r"strengthen.*commit|commit.*strengthen|weaken.*commit",
    case=False,
    na=False,
    regex=True
)

target_rows = df_filtered[mask]

if target_rows.empty:
    raise ValueError("Target strengthen/weaken question not found.")

TARGET_QID = target_rows["QID"].iloc[0]
TARGET_QUESTION = target_rows["Question Text (Short)"].iloc[0]

# Identify strengthen response
responses = df_filtered[df_filtered["QID"] == TARGET_QID]["Response"].unique()

strengthen_matches = [r for r in responses if "strengthen" in r.lower()]

if not strengthen_matches:
    raise ValueError("Strengthen response not found.")

STRENGTHEN_RESPONSE = strengthen_matches[0]

print(f"Target QID: {TARGET_QID}")
print(f"Positive class: {STRENGTHEN_RESPONSE}")

Target QID: 9
Positive class: Strengthen


In [6]:
# Clean response text for feature names
df_filtered["response_clean"] = (
    df_filtered["Response"]
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.slice(0, 40)
)

df_filtered["feature_name"] = (
    "Q" + df_filtered["QID"].astype(str) + "_" + df_filtered["response_clean"]
)

# Pivot to wide format
df_wide = (
    df_filtered
    .pivot_table(
        index=group_cols,
        columns="feature_name",
        values="Weighted Mean",
        aggfunc="first"
    )
    .reset_index()
)

n_groups = df_wide.shape[0]
n_total_columns = df_wide.shape[1]
n_features = n_total_columns - len(group_cols)

missing_pct = df_wide.isna().mean().mean() * 100

avg_responses_per_group = (
    df_filtered.groupby(group_cols)["QID"].nunique().mean()
)

print(f"Respondent groups (samples): {n_groups}")
print(f"Feature dimensionality: {n_features}")
print(f"Feature-to-sample ratio: {n_features/n_groups:.2f}")
print(f"Average questions per group: {avg_responses_per_group:.1f}")
print(f"Overall missing data: {missing_pct:.2f}%")

Respondent groups (samples): 143
Feature dimensionality: 71
Feature-to-sample ratio: 0.50
Average questions per group: 15.0
Overall missing data: 0.13%


In [7]:
# target creation
# Identify strengthen column in wide dataset
target_cols = [
    col for col in df_wide.columns
    if f"Q{int(TARGET_QID)}" in col and "strengthen" in col.lower()
]

if not target_cols:
    raise ValueError("Strengthen column not found.")

TARGET_COL = target_cols[0]

# Continuous support variable
df_wide["support_strengthen"] = df_wide[TARGET_COL]
df_wide = df_wide.dropna(subset=["support_strengthen"])

# Binary target via median split
median_support = df_wide["support_strengthen"].median()

df_wide["target"] = (
    df_wide["support_strengthen"] > median_support
).astype(int)

print(f"Median threshold: {median_support:.2f}%")
print(f"Class balance: {df_wide['target'].mean()*100:.1f}% positive")
print(f"Valid samples: {len(df_wide)}")

Median threshold: 85.00%
Class balance: 47.6% positive
Valid samples: 143


In [8]:
# Feature Engineering
# Demographic features
demographic_features = pd.get_dummies(
    df_wide[group_cols],
    drop_first=True
)

demographic_names = demographic_features.columns.tolist()

# Remove all columns related to target question (avoid leakage)
attitude_cols = [
    col for col in df_wide.columns
    if col.startswith("Q") and f"Q{int(TARGET_QID)}" not in col
]

attitude_features = df_wide[attitude_cols].fillna(
    df_wide[attitude_cols].median()
)

# Combine features
X = pd.concat([demographic_features, attitude_features], axis=1)
y = df_wide["target"].values

print("Feature Summary")
print(f"Total features: {X.shape[1]}")
print(f"Demographic features: {len(demographic_names)}")
print(f"Attitude features: {len(attitude_cols)}")

Feature Summary
Total features: 132
Demographic features: 65
Attitude features: 67


In [9]:
# Train - Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train/Test Split")
print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")


Train/Test Split
Train size: 114
Test size: 29


In [10]:
# Model Training and Evaluation

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}")

    # Logistic Regression requires scaling
    if name == "Logistic Regression":
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        X_cv = X_train_scaled

        trained_models[name] = {
            "model": model,
            "scaler": scaler
        }

    else:
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        X_cv = X_train

        trained_models[name] = {
            "model": model,
            "scaler": None
        }

    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    # Cross-validation on training data
    cv_scores = cross_val_score(
        model,
        X_cv,
        y_train,
        cv=5,
        scoring="balanced_accuracy"
    )

    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    results.append({
        "Model": name,
        "Balanced Accuracy": balanced_acc,
        "F1 Score": f1,
        "ROC AUC": auc,
        "CV Mean": cv_mean,
        "CV Std": cv_std
    })

# Create comparison table
comparison_df = pd.DataFrame(results).sort_values(
    "Balanced Accuracy",
    ascending=False
)

print("\nModel Comparison")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("model_comparison.csv", index=False)

# Select best model
best_model_name = comparison_df.iloc[0]["Model"]
best_model_info = trained_models[best_model_name]

print(f"\nBest Model Selected: {best_model_name}")


Training Logistic Regression

Training Random Forest

Training Gradient Boosting

Model Comparison
              Model  Balanced Accuracy  F1 Score  ROC AUC  CV Mean   CV Std
Logistic Regression           0.897619  0.896552 0.928571 0.685152 0.085534
      Random Forest           0.828571  0.827586 0.957143 0.754091 0.059722
  Gradient Boosting           0.795238  0.800000 0.933333 0.700909 0.038882

Best Model Selected: Logistic Regression


In [11]:
# Feature Importance using Random Forest

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_model.fit(X, y)

importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values("Importance", ascending=False)

# Save full importance table
feature_importance_df.to_csv("feature_importance.csv", index=False)

# Plot Top 20
top_features = feature_importance_df.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 20 Most Important Features")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.close()

print("Feature importance saved:")
print("- feature_importance.csv")
print("- feature_importance.png")

Feature importance saved:
- feature_importance.csv
- feature_importance.png


In [12]:
from sklearn.metrics import ConfusionMatrixDisplay

model = best_model_info["model"]
scaler = best_model_info["scaler"]

if scaler is not None:
    X_test_input = scaler.transform(X_test)
else:
    X_test_input = X_test

ConfusionMatrixDisplay.from_estimator(
    model,
    X_test_input,
    y_test,
    cmap="Blues"
)

plt.title("Confusion Matrix – Test Set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.close()

In [13]:
from sklearn.metrics import RocCurveDisplay

plt.figure()

RocCurveDisplay.from_estimator(
    model,
    X_test_input,
    y_test
)

plt.title("ROC Curve – Test Set")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=300)
plt.close()

<Figure size 1200x600 with 0 Axes>

In [15]:
# Country Clustering
# Aggregate to country level
q_cols_cluster = [col for col in df_wide.columns if col.startswith("Q")]

country_level = (
df_wide
.groupby("Country")[q_cols_cluster]
.mean()
.dropna()
.reset_index()
)

print("Country-level dataset created")
print(f"Countries: {country_level.shape[0]}")
print(f"Attitude features: {len(q_cols_cluster)}")

# Standardize features
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(country_level[q_cols_cluster])

# 
K_range = range(2, min(11, len(country_level)))
sil_scores = []
inertias = []

for k in K_range:
kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
labels = kmeans.fit_predict(X_scaled)
# 
# 


Clustering Summary
Countries: 64
Optimal clusters: 2
Best silhouette score: 0.211
Silhouette interpretation: 0.211 indicates moderate cluster separation.
Cluster outputs saved:
- country_clusters.csv
- clusters_pca.png
